# 83 · Published PT findings: the cluster view beside the continuous view

One figure per published finding. Each row has two panels on the same y scale:
- **Left, the cluster view:** what the paper's kind of method shows. Each segment cluster (S1, S2, S3) is one value
  per specimen or donor. Hollow markers are atlas donors (Census mouse, Lake human); filled markers joined by lines
  are our four specimens.
- **Right, the continuous view:** the same genes along the reconstructed pseudospace in our four specimens (S1 0–1,
  S2 1–2, S3 2–3 segment units; grey lines at the segment boundaries). A triangle under the axis marks the steepest
  point when both specimens of a species place it within 0.25 segment units of each other.

Both views use the same centring: each gene's level minus its own mean, minus the typical gene (the median gene per
segment in the cluster view, per grid point in the continuous view), then the median over a program's members (log2).

**Nothing is fitted here.** The continuous curves are notebook 81's (71's NB specimen curves); the cluster values
come from notebook 79's specimen × segment pseudobulks and notebook 82's atlas donor × segment rates.

**Selection (recorded, post hoc).** A draft of 16 candidates was drawn first. Kept: findings whose genes come from
the paper, whose cluster view in our specimens agrees with the atlases, and whose continuous curves agree between
the two specimens of the species that carries the pattern.
- **Phosphate transport** (Fernandes 2026): NaPi-IIa (Slc34a1), NaPi-IIc (Slc34a3), PiT-1 (Slc20a1). Single
  genes; the species differences replicate in Cheval and the atlases (notebook 82, section 10b).
- **CellSpectra's mouse-high S1 genes** (S-M1) and **mouse-high S3-only genes** (U-M3). These are the two programs
  whose mouse shape notebook 81 confirmed (specimen-agreed, null match 0, landmark-confirmed).

Dropped: CellSpectra's conserved functions (small amplitudes; the atlases do not confirm their peaks), Raghubar's
groups (mixed shapes), and the human S1 trough programs (they may mark old S2 units labelled S1; issue 22).

**What bounds the reading.**
- Cohort: two male control mice and two cortex sections of one male human donor.
- Within-segment positions are descriptive. The S1/S2/S3 registration gap is 0.37 segment units, the human S2 cell
  is weak, and human S3 − S1 agrees with microdissected tissue at only ρ 0.14 (issue 21).
- A mouse S1 bias can be neighbour signal (issue 24). None of the genes shown is a DCT, podocyte or renin gene,
  except S100g, one of S-M1's 96 members.

## 1 · Setup

In [ ]:
import argparse
import hashlib
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from pseudospace import published_findings as pf

T79 = results_root / 'pt_known_species_programs'                  # read only: specimen × segment pseudobulks
T81 = results_root / 'pt_published_findings'                      # read only: continuous curves and descriptors
T82 = results_root / 'pt_published_findings_external'             # read only: atlas donor × segment rates
KNOWN = project / 'docs' / 'results' / 'known-programs'
LIT = project / 'docs' / 'results' / 'published-findings'
output = results_root / 'pt_published_findings_figures'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)

SHA = {KNOWN / 'programs_hvm_members.csv': '2a3889075ca4130b102661409aee1390f1dd1f936a4171c9c9b7b0a7365299bf',
       LIT / 'programs_lit_members.csv': '8bcbd2556f52b859ad4ef62ffd2c98c14d4821e784a68ec979ecfbebbab219c7'}
REQUIRED = [*SHA, T79 / 'tables' / 'pseudobulk_rates.csv.gz', T81 / 'tables' / 'curves.npz',
            T81 / 'tables' / 'descriptors_species.csv', T81 / 'run_record.json',
            T82 / 'tables' / 'atlas_segment_rates.csv.gz', T82 / 'run_record.json']
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 79, 81 and 82 first; see data/README.md)')

NOTEBOOK_LOGIC_VERSION = '83.published_findings_figures.1'   # Bump when a calculation changes.
SEGMENTS = ('S1', 'S2', 'S3')
NAMES = ['Ctrl1A2', 'Ctrl1A4', 'HUK1_COR1', 'HUK1_MED1']
SPECIMENS = {'mouse': NAMES[:2], 'human': NAMES[2:]}
ATLAS_OF = {'mouse': 'Census mouse', 'human': 'Lake human'}
AGREE = .25                    # segment units: two specimens' steepest points this close count as one position
MIN_SLOPE = 1.                 # log2 per segment unit: a steepest point below this is not marked

FIGURES = {                    # per row: 81's curve key, panel title, genes (None: the program's frozen members)
    'fig83a_phosphate_transport': [('Slc34a1', 'Slc34a1 (NaPi-IIa)', ['Slc34a1']),
                                   ('Slc34a3', 'Slc34a3 (NaPi-IIc)', ['Slc34a3']),
                                   ('Slc20a1', 'Slc20a1 (PiT-1)', ['Slc20a1'])],
    'fig83b_cellspectra_mouse_high_s1': [('S-M1', 'CellSpectra mouse-high S1 genes (S-M1)', None)],
    'fig83c_cellspectra_mouse_high_s3_only': [('U-M3', 'CellSpectra mouse-high S3-only genes (U-M3)', None)],
}

MM = 1 / 25.4
INK, LIGHT = '#0b0b0b', '#d9d8d4'
SPECIES_COLOR = {'mouse': '#0072B2', 'human': '#D55E00'}
SPEC_COLOR = {'Ctrl1A2': '#0072B2', 'Ctrl1A4': '#56B4E9', 'HUK1_COR1': '#D55E00', 'HUK1_MED1': '#E69F00'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5, 'axes.titleweight': 'bold',
    'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': .6, 'lines.linewidth': 1.0,
    'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 90)
GUARDS = []


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')
    plt.show()
    plt.close(fig)


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


print('Results folder:', output, '· logic', NOTEBOOK_LOGIC_VERSION)

## 2 · Inputs and guards (not cached)

- **G1:** the program member files match their frozen sha256.
- **G2:** notebooks 81 and 82 ran at logic versions 81.* and 82.*; 81's curves hold every row × specimen on one grid
  that spans S1–S3.
- **G3:** 79's pseudobulks hold S1, S2 and S3 for the four specimens; 82's atlas rates hold Census mouse (6 donors)
  and Lake human (7 donors) with S1, S2 and S3.

In [ ]:
for path, digest in SHA.items():
    assert sha256(path) == digest, f'G1: {path.name} differs from its frozen sha256'
members_all = pd.concat([pd.read_csv(KNOWN / 'programs_hvm_members.csv'), pd.read_csv(LIT / 'programs_lit_members.csv')])
MEMBERS = {p: g.gene.astype(str).tolist() for p, g in members_all.groupby('program_id')}
GUARDS.append(f'G1: {len(SHA)} member files match their sha256')

run81 = json.loads((T81 / 'run_record.json').read_text())
run82 = json.loads((T82 / 'run_record.json').read_text())
assert str(run81['notebook_logic_version']).startswith('81.') and str(run82['notebook_logic_version']).startswith('82.')
with np.load(T81 / 'tables' / 'curves.npz', allow_pickle=False) as payload:
    CURVES = {k: payload[k] for k in payload.files}
UNITS = CURVES['s']
assert np.all(np.diff(UNITS) > 0) and UNITS[0] < .25 and UNITS[-1] > 2.75, (UNITS[0], UNITS[-1])
ROWS = [(stem, title, genes if genes is not None else MEMBERS[key], key)
        for stem, rows in FIGURES.items() for key, title, genes in rows]
for _, title, _, key in ROWS:
    for spec in NAMES:
        assert f'{key}|{spec}' in CURVES, f'G2: no 81 curve for {key} in {spec}'
GUARDS.append(f"G2: 81 ({run81['notebook_logic_version']}) and 82 ({run82['notebook_logic_version']}); "
              f'{len(ROWS)} rows × 4 specimens of curves on {len(UNITS)} grid points ({UNITS[0]:.2f}–{UNITS[-1]:.2f})')

RATES79 = pd.read_csv(T79 / 'tables' / 'pseudobulk_rates.csv.gz', index_col=0, header=[0, 1])
OURS_RATES = {spec: pd.DataFrame({s: RATES79[(s, spec)] for s in SEGMENTS}) for spec in NAMES}
ATLAS = pd.read_csv(T82 / 'tables' / 'atlas_segment_rates.csv.gz', index_col=0, header=[0, 1, 2])
ATLAS_RATES = {}
for species, name in ATLAS_OF.items():
    block = ATLAS[name]
    donors = list(dict.fromkeys(block.columns.get_level_values(0)))
    ATLAS_RATES[species] = {d: block[d][list(SEGMENTS)] for d in donors}
assert len(ATLAS_RATES['mouse']) == 6 and len(ATLAS_RATES['human']) == 7, {k: len(v) for k, v in ATLAS_RATES.items()}
GUARDS.append(f"G3: 79's pseudobulks for {len(NAMES)} specimens × S1/S2/S3 ({len(RATES79):,} genes); atlas donors "
              f"{ {k: len(v) for k, v in ATLAS_RATES.items()} }")
print('\n'.join('· ' + g for g in GUARDS))

## 3 · The two views and their readout

Per row: the cluster values (`pf.segment_profile`) for every specimen and atlas donor, and per specimen the
within-segment change and the steepest point of the continuous curve (`pf.within_segment_change`). A species' steepest
point is reported when both specimens place it within 0.25 segment units and its slope is at least 1 log2 per
segment unit.

In [ ]:
CLUSTER, CHANGE, STEEPEST = {}, {}, {}
rows_out = []
for stem, title, genes, key in ROWS:
    CLUSTER[(title, 'ours')] = pd.DataFrame({spec: pf.segment_profile(OURS_RATES[spec], genes) for spec in NAMES}).T
    for species in ('mouse', 'human'):
        CLUSTER[(title, species)] = pd.DataFrame({d: pf.segment_profile(r, genes)
                                                  for d, r in ATLAS_RATES[species].items()}).T
    for spec in NAMES:
        CHANGE[(title, spec)] = pf.within_segment_change(CURVES[f'{key}|{spec}'], UNITS)
    for species, specs in SPECIMENS.items():
        a, b = (CHANGE[(title, s)] for s in specs)
        agreed = (abs(a['steepest position'] - b['steepest position']) <= AGREE
                  and np.sign(a['steepest slope']) == np.sign(b['steepest slope'])
                  and min(abs(a['steepest slope']), abs(b['steepest slope'])) >= MIN_SLOPE)
        STEEPEST[(title, species)] = (a['steepest position'] + b['steepest position']) / 2 if agreed else np.nan
        ours = CLUSTER[(title, 'ours')].loc[specs].mean()
        atlas = CLUSTER[(title, species)].median()
        row = {'figure': stem, 'row': title, 'members': len(genes), 'species': species}
        row.update({f'cluster ours {s}': ours[s] for s in SEGMENTS})
        row.update({f'cluster atlas {s}': atlas[s] for s in SEGMENTS})
        row.update({f'within {s} (spec 1, spec 2)': f'{a[s]:+.2f}, {b[s]:+.2f}' for s in SEGMENTS})
        row.update({'steepest (spec 1, spec 2)': f"{pf.segment_position(a['steepest position'])} ({a['steepest slope']:+.1f}), "
                                                 f"{pf.segment_position(b['steepest position'])} ({b['steepest slope']:+.1f})",
                    'steepest agreed': pf.segment_position(STEEPEST[(title, species)]) if agreed else 'no'})
        rows_out.append(row)
READOUT = pd.DataFrame(rows_out)
display(READOUT.round(2))
desc81 = pd.read_csv(T81 / 'tables' / 'descriptors_species.csv', index_col=0)
keys = [key for _, _, _, key in ROWS]
display(desc81.loc[keys, ['mouse shape', 'mouse position (segment)', 'mouse switch', 'human shape',
                          'human position (segment)', 'across species']
                   + [c for c in ('mouse null match', 'human null match') if c in desc81]])

## 4 · Figures

Colours: dark and light blue are the two mice (Ctrl1A2, Ctrl1A4); orange and yellow the two human sections
(HUK1_COR1, HUK1_MED1). Hollow blue markers are Census mouse donors; hollow orange markers are Lake human donors.

In [ ]:
def side_by_side(stem, rows):
    fig, axes = plt.subplots(len(rows), 2, figsize=(120 * MM, 38 * MM * len(rows)), squeeze=False,
                             gridspec_kw={'width_ratios': [1, 2.6], 'wspace': .12, 'hspace': .55}, sharey='row')
    for (left, right), (key, title, _) in zip(axes, rows):
        for species, offset in (('mouse', -.14), ('human', .14)):
            atlas = CLUSTER[(title, species)]
            for k, seg in enumerate(SEGMENTS):
                left.scatter(np.full(len(atlas), k + offset), atlas[seg], s=7, facecolors='none',
                             edgecolors=SPECIES_COLOR[species], linewidths=.5, alpha=.7, zorder=2)
        ours = CLUSTER[(title, 'ours')]
        for spec in NAMES:
            left.plot(range(3), ours.loc[spec, list(SEGMENTS)].to_numpy(float), marker='o', ms=2.5,
                      color=SPEC_COLOR[spec], lw=.8, zorder=3)
            right.plot(UNITS, CURVES[f'{key}|{spec}'], color=SPEC_COLOR[spec], lw=.9, zorder=3)
        for b in (1, 2):
            right.axvline(b, color=LIGHT, lw=.6, zorder=0)
        for ax in (left, right):
            ax.axhline(0, color=LIGHT, lw=.5, zorder=0)
        left.set_xticks(range(3))
        left.set_xticklabels(SEGMENTS)
        left.set_xlim(-.45, 2.45)
        right.set_xlim(0, 3)
        right.set_xticks([0, .5, 1, 1.5, 2, 2.5, 3])
        left.set_title(title, loc='left')
        left.set_ylabel('log2, centred')
        lo, hi = right.get_ylim()
        for species in ('mouse', 'human'):
            x = STEEPEST[(title, species)]
            if np.isfinite(x):
                right.scatter([x], [lo + .02 * (hi - lo)], marker='^', s=14, color=SPECIES_COLOR[species], zorder=4,
                              clip_on=False)
        right.set_ylim(lo, hi)
    axes[-1, 0].set_xlabel('segment cluster')
    axes[-1, 1].set_xlabel('pseudospace (segment units)')
    save(fig, stem)


for stem, rows in FIGURES.items():
    side_by_side(stem, rows)

## 5 · Outputs and run record

In [ ]:
READOUT.to_csv(tables / 'readout.csv', index=False)
cluster_out = pd.concat({f'{t}|{w}': v for (t, w), v in CLUSTER.items()}, names=['row|source', 'unit'])
cluster_out.to_csv(tables / 'cluster_values.csv')
record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'inputs': {'81': run81['notebook_logic_version'],
          '82': run82['notebook_logic_version']}, 'sha256': {p.name: d for p, d in SHA.items()}, 'guards': GUARDS,
          'figures': {stem: [t for _, t, _ in rows] for stem, rows in FIGURES.items()},
          'steepest_agreed': {f'{t}|{s}': (None if not np.isfinite(v) else round(float(v), 3)) for (t, s), v in STEEPEST.items()},
          'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(record, indent=1, default=str))
print('Tables:', sorted(p.name for p in tables.iterdir()), '· figures:', sorted(p.name for p in figures.glob('*.png')))
print(f"Run record written · {record['seconds']} s")